# ☀️ Solar Filament Segmentation 2026: Baseline Pipeline (U-Net)
Welcome to the Baseline Pipeline for the Solar Filament Segmentation Challenge!

## 📌 Overview & Architecture
This notebook provides a complete, start-to-finish PyTorch pipeline for solar filament segmentation.

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import sys
import json
import time
import warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Reduces fragmentation-related OOMs on long runs (suggested directly in the CUDA OOM error message)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# --------------------------------------------------------------------------------
# OFFLINE-SAFE DEPENDENCY HANDLING
# --------------------------------------------------------------------------------
SMP_AVAILABLE = False
try:
    import segmentation_models_pytorch as smp
    SMP_AVAILABLE = True
except ImportError:
    print("segmentation_models_pytorch not preinstalled -- attempting a quick pip install "
          "(short timeout so an offline kernel fails fast instead of hanging)...")
    exit_code = os.system(
        "pip install -q --timeout 8 --retries 1 segmentation-models-pytorch"
    )
    if exit_code == 0:
        try:
            import segmentation_models_pytorch as smp
            SMP_AVAILABLE = True
        except ImportError:
            pass
    if not SMP_AVAILABLE:
        print("Internet is OFF (or the package truly isn't reachable) -- "
              "falling back to a built-in, dependency-free UNet (torchvision resnet34 "
              "encoder) defined below. No external package or download required.")

PYCOCOTOOLS_AVAILABLE = False
try:
    import pycocotools.mask as mask_util
    PYCOCOTOOLS_AVAILABLE = True
except ImportError:
    print("pycocotools not preinstalled -- RLE encoding will use the built-in "
          "pure-Python COCO-RLE encoder defined below (produces byte-identical "
          "output to pycocotools, verified against it).")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    raise RuntimeError(
        "albumentations is missing and is NOT optional for this notebook (used for "
        "train/val/test transforms). It ships preinstalled on Kaggle's standard Python "
        "GPU image, so this only happens on a custom/minimal image -- add it as a pip "
        "requirement on an Internet-ON run, or attach an offline wheel dataset."
    )


class _ConvBlock(nn.Module):
    """Two Conv-BN-ReLU layers in a row -- the standard UNet decoder building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetResNet34Fallback(nn.Module):
    """Standard UNet decoder over a torchvision resnet34 encoder.
    forward(x) -> raw logits of shape (B, 1, H, W), same contract as the
    smp.UnetPlusPlus(activation=None) model it replaces. Used automatically
    when segmentation_models_pytorch is unavailable -- see build_model()."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = torchvision.models.resnet34(weights=weights)
        except Exception:
            print("Could not fetch ImageNet weights for the fallback encoder (offline) "
                  "-- using random init instead. Training will still run, just from scratch.")
            resnet = torchvision.models.resnet34(weights=None)

        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # /2,  64ch
        self.pool0 = resnet.maxpool                                       # /4
        self.enc1 = resnet.layer1                                         # /4,  64ch
        self.enc2 = resnet.layer2                                         # /8,  128ch
        self.enc3 = resnet.layer3                                         # /16, 256ch
        self.enc4 = resnet.layer4                                         # /32, 512ch

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = _ConvBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = _ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = _ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = _ConvBlock(128, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec0 = _ConvBlock(32, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.enc0(x)
        p0 = self.pool0(e0)
        e1 = self.enc1(p0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))
        return self.head(d0)


def pure_python_coco_rle(binary_mask):
    """Fallback RLE encoder when pycocotools is unavailable. Produces the SAME
    compressed COCO-RLE ASCII 'counts' format as pycocotools.mask.encode --
    verified byte-identical across 200+ random masks. (NOTE: this is
    deliberately NOT the plain space-separated "start length" Kaggle RLE
    format -- this notebook's submission column is 'segmentation_rle' and
    matches the COCO-style encoding pycocotools produces, so the fallback
    must match that exact format or a mix of pycocotools-encoded and
    fallback-encoded rows would be inconsistent.)"""
    pixels = np.asarray(binary_mask, dtype=np.uint8).flatten(order='F')
    counts = []
    prev, run = 0, 0
    for p in pixels:
        if p == prev:
            run += 1
        else:
            counts.append(run)
            run = 1
            prev = p
    counts.append(run)

    out = []
    for i, val in enumerate(counts):
        x = val - counts[i - 2] if i > 2 else val
        more = True
        while more:
            c = x & 0x1f
            x >>= 5
            more = (x != -1) if (c & 0x10) else (x != 0)
            if more:
                c |= 0x20
            out.append(chr(c + 48))
    return ''.join(out)


def mask_to_coco_rle(binary_mask):
    """Uses pycocotools if available; otherwise the pure-Python fallback above,
    which produces byte-identical output."""
    if PYCOCOTOOLS_AVAILABLE:
        fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
        rle = mask_util.encode(fortran_mask)
        rle['counts'] = rle['counts'].decode('utf-8')
        return rle['counts']
    else:
        return pure_python_coco_rle(binary_mask)


def predict_with_tta(model, images):
    """Average predictions over identity + h-flip + v-flip + 180-rotate.
    Cheap accuracy gain (no extra training), at ~4x inference cost."""
    with torch.no_grad():
        p0 = torch.sigmoid(model(images))
        p1 = torch.sigmoid(model(torch.flip(images, dims=[3]))).flip(dims=[3])
        p2 = torch.sigmoid(model(torch.flip(images, dims=[2]))).flip(dims=[2])
        p3 = torch.sigmoid(model(torch.flip(images, dims=[2, 3]))).flip(dims=[2, 3])
    return (p0 + p1 + p2 + p3) / 4.0


def get_tile_coords(h, w, tile_size, overlap):
    """Top-left (x, y) grid of tile origins that fully covers an h x w image,
    with the last tile in each row/column pulled in to stay flush with the
    edge (so tiles never run off the image and every pixel is covered)."""
    stride = max(tile_size - overlap, 1)
    xs = list(range(0, max(w - tile_size, 0) + 1, stride))
    ys = list(range(0, max(h - tile_size, 0) + 1, stride))
    if not xs or xs[-1] + tile_size < w:
        xs.append(max(w - tile_size, 0))
    if not ys or ys[-1] + tile_size < h:
        ys.append(max(h - tile_size, 0))
    return sorted(set(xs)), sorted(set(ys))


def sliding_window_predict(model, image_tensor, tile_size=640, overlap=128, device="cuda"):
    """
    image_tensor: normalized (C, H, W) tensor for ONE full-resolution test image.
    Runs TTA-averaged prediction on each overlapping tile and stitches them into
    a single (1, 1, H, W) probability map, averaging predictions in overlap regions.
    """
    c, h, w = image_tensor.shape
    pad_h, pad_w = max(tile_size - h, 0), max(tile_size - w, 0)
    x = F.pad(image_tensor.unsqueeze(0), (0, pad_w, 0, pad_h), mode="reflect")
    _, _, hp, wp = x.shape
    prob_sum = torch.zeros((1, 1, hp, wp), device=device)
    weight = torch.zeros_like(prob_sum)
    xs, ys = get_tile_coords(hp, wp, tile_size, overlap)
    for y in ys:
        for x0 in xs:
            tile = x[:, :, y:y+tile_size, x0:x0+tile_size].to(device)
            prob = predict_with_tta(model, tile)
            prob_sum[:, :, y:y+tile_size, x0:x0+tile_size] += prob
            weight[:, :, y:y+tile_size, x0:x0+tile_size] += 1
    return (prob_sum / weight.clamp_min(1))[:, :, :h, :w]


# ============================================================================
# GLOBAL CONFIGURATION
# ============================================================================
CFG = {
    "seed": 42,
    "patch_size": 640,           # native-resolution crop size for both training AND validation
                                  # patches -- NOT a resize, so thin filament barbs never get
                                  # shrunk away like they would resizing a full ~2048px image down.
    "pos_patch_prob": 0.85,      # probability a TRAINING crop is centered on a filament pixel
                                  # rather than a uniformly random (usually empty) crop.
    "val_pos_patch_prob": 0.85,  # NEW -- was 0.5. Raised to match training. With 0.5, roughly
                                  # half the fixed validation patches were random crops that are
                                  # almost always empty of filament content (filaments cover well
                                  # under 1% of pixels), which -- combined with the val-dice
                                  # averaging bug fixed below -- was a major source of the val
                                  # metric being inflated relative to real leaderboard performance.
    "batch_size": 2,              # physical batch size actually placed on GPU per forward pass
    "accum_steps": 4,             # gradient accumulation -> effective batch 2*4=8 for the GRADIENT
                                   # estimate, while GPU memory only ever holds 2 samples at once.
                                   # Does NOT fix BatchNorm noise (each forward pass still only
                                   # sees 2 real images for BN stats) -- that's what freeze_bn is for.
    "use_amp": torch.cuda.is_available(),
    "epochs": 25,
    "lr": 3e-4,
    "backbone": "se_resnext50_32x4d",   # swap to resnet34 / efficientnet-b0 if inference-time
                                          # budget is tight -- efficiency is 70% of this
                                          # competition's score, alongside Dice.
    "encoder_weights": "imagenet",
    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "threshold": 0.40,
    "cldice_weight": 0.3,        # weight of the clDice (centerline/connectivity) loss term
    "warmup_epochs": 2,          # epochs of plain pos-weighted BCE before switching to the
                                  # harder FocalTversky+clDice loss -- gets the model predicting
                                  # *something* foreground-shaped first.
    "bce_pos_weight": 50.0,      # positive-class weight for the warm-up BCE.
    "lr_patience": 3,            # ReduceLROnPlateau: halve LR if Val Dice plateaus this many epochs.
    "lr_factor": 0.5,
    "clip_grad_norm": 1.0,       # max gradient norm -- a safety net against occasional large/spiky
                                  # gradients knocking training off course. None disables it.
    "freeze_bn": True,           # freeze BatchNorm running-stats to eval mode (see set_bn_eval
                                  # below) -- batch_size=2 is too small for BatchNorm to estimate
                                  # reliable per-batch statistics, a well-documented small-batch
                                  # fine-tuning instability.
}

def seed_everything(seed=42):
    """Seeds every RNG the MAIN process touches. Does NOT by itself make
    DataLoader worker processes (num_workers>0) properly independent -- see
    seed_worker() below and its wiring via worker_init_fn in the training cell."""
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    """
    Passed as DataLoader's worker_init_fn. WHY: with num_workers>0, PyTorch
    forks separate worker processes, each starting with an EXACT COPY of the
    parent's numpy/random RNG state at fork time. Without reseeding here,
    multiple workers can produce IDENTICAL "random" sequences within the same
    epoch -- silently reducing effective data diversity and correlating
    supposedly-independent workers. torch.initial_seed() gives each worker a
    PyTorch-derived unique seed (via the DataLoader's `generator`); we just
    also need to feed that into numpy's and Python's own RNGs, since our code
    (sample_patch) and some albumentations internals use those, not torch's.
    """
    import random
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

seed_everything(CFG["seed"])
g_generator = torch.Generator().manual_seed(CFG["seed"])
print("Device:", CFG["device"], "| SMP available:", SMP_AVAILABLE, "| pycocotools available:", PYCOCOTOOLS_AVAILABLE)

segmentation_models_pytorch not preinstalled -- attempting a quick pip install (short timeout so an offline kernel fails fast instead of hanging)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
Device: cuda | SMP available: True | pycocotools available: True


## 1. Paths & Annotation Helper Functions 📁¶
Loads the MAGFiLO annotations (standard COCO format) and builds a `filename -> [annotation dicts]` lookup.

In [2]:
# ============================================================================
# PATHS & ANNOTATION LOADING
# ============================================================================
# The annotations file is standard COCO format (top-level keys: info, licenses,
# categories, images, annotations) -- confirmed directly by the mask sanity-check
# cell earlier. We build a filename -> [annotation dicts] lookup from images +
# annotations, rather than assuming the JSON is already keyed by filename (an
# earlier version of this notebook made that wrong assumption, which silently
# produced empty masks for every image).
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

BASE_DIR = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/")
TRAIN_IMG_DIR = BASE_DIR / "train/train_images"
TEST_IMG_DIR = BASE_DIR / "test/test_images"
ANNOTATIONS_PATH = BASE_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# 1. Load Raw COCO JSON
with open(ANNOTATIONS_PATH, 'r') as f:
    coco_data = json.load(f)

# 2. Build COCO Lookup Index: image_id -> file_name, then file_name -> [annotations]
file_to_id = {img['file_name']: img['id'] for img in coco_data['images']}

id_to_anns = defaultdict(list)
for ann in coco_data['annotations']:
    id_to_anns[ann['image_id']].append(ann)

annotations_data = {
    fn: id_to_anns[file_to_id[fn]]
    for fn in file_to_id
}

# 3. Create DataFrames & Splits
train_files = sorted([p.name for p in TRAIN_IMG_DIR.glob("*.jpeg")])
test_files = sorted([p.name for p in TEST_IMG_DIR.glob("*.jpeg")])

df_train = pd.DataFrame({'filename': train_files})
df_test = pd.DataFrame({'filename': test_files})

train_df, val_df = train_test_split(df_train, test_size=0.2, random_state=CFG['seed'], shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total Train Images: {len(train_df)} | Validation Images: {len(val_df)} | Test Images: {len(df_test)}")
print(f"Indexed annotations for {len(annotations_data)} images in COCO dictionary.")

Total Train Images: 565 | Validation Images: 142 | Test Images: 180
Indexed annotations for 707 images in COCO dictionary.


## 2. Dataset & Mask Generation Pipeline 🧬¶
Converts COCO polygon/RLE segmentation into binary ground-truth masks, and defines the training vs.
fixed-patch validation datasets.

In [3]:
# ============================================================================
# MASK GENERATION + DATASETS
# ============================================================================
def create_mask_from_annotation(annotations, shape):
    """Convert all COCO polygon/RLE annotations for one image into one binary mask."""
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for ann in annotations or []:
        seg = ann.get("segmentation", [])
        if isinstance(seg, list):
            # Polygon-style: list of one or more flat coordinate lists.
            for poly in seg:
                pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
                if len(pts) >= 3:
                    cv2.fillPoly(mask, [pts], 1)
        elif isinstance(seg, dict):
            # COCO RLE-style segmentation.
            if not PYCOCOTOOLS_AVAILABLE:
                raise RuntimeError("RLE annotation encountered but pycocotools is unavailable. Install pycocotools before training.")
            rle = dict(seg)
            decoded = mask_util.decode(rle)
            if decoded.ndim == 3:
                decoded = decoded.max(axis=2)
            mask = np.maximum(mask, decoded.astype(np.uint8))
    return mask


import os
import cv2
from torch.utils.data import Dataset


def sample_patch(image, mask, patch_size=640, pos_prob=0.85, rng=None):
    """
    Random-crop a (patch_size x patch_size) patch at NATIVE resolution.

    rng: defaults to the np.random module (normal random behavior, used for
    TRAINING). Pass a np.random.RandomState(seed=...) instead to get a
    REPRODUCIBLE patch -- this is what makes FixedPatchSolarDataset below
    produce the SAME validation crop every epoch instead of a different
    random one each time.
    """
    if rng is None:
        rng = np.random
    h, w = image.shape[:2]
    if h < patch_size or w < patch_size:
        pad_h, pad_w = max(0, patch_size-h), max(0, patch_size-w)
        image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
        mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
        h, w = image.shape[:2]
    has_pos = bool(mask.any())
    if has_pos and rng.rand() < pos_prob:
        # Center the crop on a randomly chosen filament pixel, so rare/thin
        # structures aren't starved out by mostly-empty random crops.
        ys, xs = np.where(mask > 0)
        j = rng.randint(len(ys))
        cy, cx = int(ys[j]), int(xs[j])
        top = int(np.clip(cy - patch_size//2, 0, h-patch_size))
        left = int(np.clip(cx - patch_size//2, 0, w-patch_size))
    else:
        top = rng.randint(0, h-patch_size+1)
        left = rng.randint(0, w-patch_size+1)
    return image[top:top+patch_size, left:left+patch_size], mask[top:top+patch_size, left:left+patch_size]


class SolarDataset(Dataset):
    """Training dataset: a NEW random patch is sampled on every __getitem__ call
    (so the same image contributes a different crop each epoch)."""
    def __init__(self, df, img_dir, annotations, transform=None, patch_size=640, pos_patch_prob=0.85):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir; self.annotations=annotations
        self.transform=transform; self.patch_size=patch_size; self.pos_patch_prob=pos_patch_prob
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        filename=self.df.iloc[idx]['filename']
        image=cv2.imread(str(Path(self.img_dir)/filename))
        if image is None: raise FileNotFoundError(filename)
        image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask=create_mask_from_annotation(self.annotations.get(filename, []), image.shape[:2])
        image, mask=sample_patch(image, mask, self.patch_size, self.pos_patch_prob, rng=np.random)
        if self.transform:
            out=self.transform(image=image, mask=mask); image, mask=out['image'], out['mask']
        mask=torch.as_tensor(np.ascontiguousarray(mask), dtype=torch.float32).unsqueeze(0)
        return image, mask


class FixedPatchSolarDataset(Dataset):
    """
    Validation dataset: exactly ONE deterministic native-resolution crop per
    image, fixed by a per-image seed (self.seed + idx) so it's identical every
    epoch -- Val Dice changes then reflect the model actually improving, not a
    different random crop happening to be easier or harder this time.

    pos_patch_prob is now passed in matching CFG['val_pos_patch_prob'] (0.85,
    same as training) -- previously this was hardcoded lower (0.5), meaning
    close to half the validation patches were near-empty random crops. See
    compute_val_dice's docstring below for why that combination (low
    pos_patch_prob + per-image score averaging) was inflating the reported
    Val Dice well above real leaderboard performance.
    """
    def __init__(self, df, img_dir, annotations, transform=None, patch_size=640, seed=42, pos_patch_prob=0.85):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir; self.annotations=annotations
        self.transform=transform; self.patch_size=patch_size; self.seed=seed; self.pos_patch_prob=pos_patch_prob
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        filename=self.df.iloc[idx]['filename']
        image=cv2.imread(str(Path(self.img_dir)/filename))
        if image is None: raise FileNotFoundError(filename)
        image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask=create_mask_from_annotation(self.annotations.get(filename, []), image.shape[:2])
        rng=np.random.RandomState(self.seed + idx)   # fixed seed -> identical patch every epoch
        image, mask=sample_patch(image, mask, self.patch_size, self.pos_patch_prob, rng=rng)
        if self.transform:
            out=self.transform(image=image, mask=mask); image, mask=out['image'], out['mask']
        mask=torch.as_tensor(np.ascontiguousarray(mask), dtype=torch.float32).unsqueeze(0)
        return image, mask

### 2b. Mask sanity check 🔍
Confirms ground-truth masks are actually non-empty BEFORE training.

In [4]:
# ============================================================================
# 2b. MASK SANITY CHECK
# ============================================================================
# Confirms ground-truth masks are actually non-empty BEFORE training, and
# surfaces the raw annotation structure directly so any future format
# mismatch is diagnosed here instead of discovered after a full training run.

sample_fn = train_df.iloc[0]['filename']
ann_keys_sample = list(annotations_data.keys())[:5]
print(f"Sample train filename: {sample_fn!r}")
print(f"First 5 annotation keys: {ann_keys_sample}")
print(f"Sample filename found directly as an annotation key: {sample_fn in annotations_data}")
if sample_fn not in annotations_data:
    stem = Path(sample_fn).stem
    stem_matches = [k for k in annotations_data.keys() if Path(k).stem == stem]
    print(f"⚠️  Direct key lookup failed. Keys matching by filename stem ({stem!r}): {stem_matches[:3]}")
    if stem_matches:
        print("    -> Looks like a file-extension or path-prefix mismatch, not a parsing bug.")
        print("       Fix the lookup (e.g. match on Path(fn).stem) rather than the mask parser.")

print(f"\nRaw annotation entry for {sample_fn!r} (inspect this to confirm the parser matches it):")
print(repr(annotations_data.get(sample_fn, annotations_data.get(Path(sample_fn).stem, "NOT FOUND")))[:800])
print()

sample_n = min(40, len(train_df))
nonempty_count = 0
frac_list = []
for i in range(sample_n):
    fn = train_df.iloc[i]['filename']
    img = cv2.imread(str(TRAIN_IMG_DIR / fn))
    h, w = img.shape[:2]
    ann_entry = annotations_data.get(fn, [])
    m = create_mask_from_annotation(ann_entry, (h, w))
    if m.sum() > 0:
        nonempty_count += 1
    frac_list.append(m.sum() / (h * w))

mean_frac = float(np.mean(frac_list)) if frac_list else 0.0
print(f"Sampled {sample_n} training images:")
print(f"  Non-empty masks: {nonempty_count}/{sample_n}")
print(f"  Mean foreground pixel fraction: {mean_frac*100:.4f}%")

if nonempty_count == 0:
    print("BUG STILL PRESENT. Check the filename-key check above first -- if that's clean, the raw "
          "annotation entry printed above needs manual inspection.")
elif mean_frac < 0.001:
    print("Masks are non-empty but filaments are an extremely small fraction of pixels "
          "(<0.1%) -- this is a genuine, severe class-imbalance problem. The warm-up phase "
          "added to the training loop below is aimed squarely at this.")
else:
    print("Masks look fine now. If Val Dice was still 0.0000 / frozen before this fix, that was "
          "downstream of the empty-mask bug, not a separate training issue -- re-run training below.")

Sample train filename: '20120917142654Bh.jpeg'
First 5 annotation keys: ['20140609195854Bh.jpeg', '20111116063134Lh.jpeg', '20130725122934Ch.jpeg', '20141027025054Uh.jpeg', '20160714235434Lh.jpeg']
Sample filename found directly as an annotation key: True

Raw annotation entry for '20120917142654Bh.jpeg' (inspect this to confirm the parser matches it):
[{'segmentation': [[540.8657, 1158.9766, 540.4257, 1157.3566, 538.6488, 1155.5797, 534.6411, 1153.349, 530.0, 1154.0, 524.0, 1157.0, 511.0, 1157.0, 510.0, 1156.0, 507.0, 1156.0, 506.0, 1154.0, 503.0346, 1152.0231, 494.0518, 1151.4262, 487.0828, 1149.3655, 484.9652, 1147.8448, 482.3076, 1147.5748, 481.441, 1145.3508, 478.1293, 1142.7996, 472.0, 1136.0, 470.0, 1136.0, 464.9024, 1132.9024, 448.6872, 1131.1777, 441.6951, 1128.947, 439.9182, 1127.1701, 439.5821, 1125.1675, 435.7261, 1123.5749, 427.3833, 1114.6352, 423.8019, 1109.263, 422.765, 1104.6585, 423.1694, 1103.1694, 419.0, 1101.0, 417.0, 1101.0, 414.2786, 1104.6285, 420.4483, 1111.395

## 3. Model Architecture & Loss Function 🏗️¶
U-Net++ (SE-ResNeXt50 encoder) when available, dependency-free fallback UNet if offline.
Loss = FocalTversky + weighted **real** soft clDice (restored from a placeholder this version --
see the comment block in the cell below), with a pos-weighted BCE warm-up phase first.

**Also fixed this version:** `compute_val_dice` switched from a per-patch macro-average (which was
handing out "free" near-1.0 scores to trivially-empty validation patches) to a global/micro-average,
and `val_pos_patch_prob` raised to match training -- together these were likely the dominant cause
of the 0.62 (Val Dice) vs. 0.18 (actual leaderboard) gap seen last run.

In [5]:
# ============================================================================
# 3. MODEL ARCHITECTURE & LOSS FUNCTION
# ============================================================================
import torch
import torch.nn as nn

# ---- BatchNorm freezing helper ----
def set_bn_eval(module):
    """
    Recursively sets BatchNorm layers to eval() mode -- makes them use their
    stable, pretrained running mean/variance instead of re-estimating
    statistics from the current mini-batch. Learnable scale/shift (weight/bias)
    parameters are UNAFFECTED and keep training normally via backprop.

    WHY: batch_size=2 is far too small for BatchNorm to estimate a reliable
    per-batch mean/variance (gradient accumulation doesn't help this -- each
    individual forward pass still only sees 2 real images). Noisy BatchNorm
    statistics are a well-documented cause of training instability when
    fine-tuning pretrained CNNs with small batches.

    IMPORTANT: model.train() resets EVERY submodule -- including BatchNorm --
    back to training mode. model.apply(set_bn_eval) must be called again every
    single epoch, right after model.train() -- see the training loop below.
    """
    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
        module.eval()


# ---- Model builder ----
def build_model():
    """Builds U-Net++ (segmentation_models_pytorch) when available, or the
    dependency-free fallback UNet (torchvision resnet34 encoder) otherwise."""
    if SMP_AVAILABLE:
        return smp.UnetPlusPlus(
            encoder_name=CFG['backbone'],
            encoder_weights=CFG['encoder_weights'],
            in_channels=3, classes=1, activation=None
        )
    return UNetResNet34Fallback(pretrained=True)


# ============================================================================
# LOSS 1: Focal Tversky
# Handles heavy background/filament class imbalance. beta > alpha biases
# toward recall (penalizes missed filament pixels harder than false
# positives). gamma applies a "focal" reweighting that pushes harder on
# already-hard examples.
# ============================================================================
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=4/3, smooth=1e-6):
        super(FocalTverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)

        tp = (probs * targets).sum()
        fp = ((1 - targets) * probs).sum()
        fn = (targets * (1 - probs)).sum()

        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma
        return focal_tversky


# ============================================================================
# LOSS 2: soft clDice (Shit et al., CVPR 2021) -- RESTORED, was a placeholder
# ============================================================================
# WHAT WAS WRONG: the previous version of soft_cldice() was a placeholder that
# just computed 1 - plain_pixel_dice(pred, gt) -- i.e. it used ONLY
# pred.sum(), gt.sum(), and (pred*gt).sum(). No spatial or morphological
# information at all. That means it is MATHEMATICALLY INCAPABLE of
# distinguishing "one continuous filament" from "the same pixels broken into
# several disconnected fragments" -- two predictions with identical pixel
# overlap get IDENTICAL loss from the placeholder, regardless of connectivity.
# The competition rubric explicitly penalizes exactly that fragmentation
# (one-to-many / many-to-one matching), so training without a real
# connectivity signal likely produced predictions that look fine pixel-wise
# but score worse on the actual instance-matching metric than Val Dice
# suggested -- part of why Val Dice (0.62) and the leaderboard score (0.18)
# diverged so much. This restores the real, differentiable soft-skeleton
# version.
def soft_erode(x):
    """Approximate morphological erosion via negated max-pooling."""
    return -F.max_pool2d(-x, kernel_size=3, stride=1, padding=1)

def soft_dilate(x):
    """Approximate morphological dilation via max-pooling."""
    return F.max_pool2d(x, kernel_size=3, stride=1, padding=1)

def soft_open(x):
    """Morphological opening = erode then dilate -- strips thin protrusions."""
    return soft_dilate(soft_erode(x))

def soft_skeletonize(x, iters=10):
    """Iteratively peels off the 'outer layer' (x minus its opening) to
    approximate a 1-pixel-wide skeleton/centerline, using only differentiable
    ops so gradients can flow back through it during training."""
    x1 = soft_open(x)
    skel = F.relu(x - x1)
    for _ in range(iters):
        x = soft_erode(x)
        x1 = soft_open(x)
        delta = F.relu(x - x1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

def soft_cldice(probs, targets, iters=15, smooth=1e-5):
    """clDice: harmonic mean of (a) precision of the predicted skeleton
    against the true mask, and (b) sensitivity of the true skeleton against
    the predicted mask. High only when the predicted centerline actually lies
    inside the true filament AND the true centerline is actually covered by
    the prediction -- i.e. it specifically rewards staying one connected
    thread, not just covering the right area."""
    skel_pred = soft_skeletonize(probs, iters)
    skel_true = soft_skeletonize(targets, iters)
    tprec = (torch.sum(skel_pred * targets) + smooth) / (torch.sum(skel_pred) + smooth)
    tsens = (torch.sum(skel_true * probs) + smooth) / (torch.sum(skel_true) + smooth)
    return 1.0 - 2.0 * (tprec * tsens) / (tprec + tsens + smooth)


class CombinedLoss(nn.Module):
    """The actual training loss (post-warmup): FocalTversky + weighted REAL soft clDice."""
    def __init__(self, cldice_weight=0.3, cldice_iters=15):
        super(CombinedLoss, self).__init__()
        self.focal_tversky = FocalTverskyLoss()
        self.cldice_weight = cldice_weight
        self.cldice_iters = cldice_iters

    def forward(self, logits, targets):
        ft_loss = self.focal_tversky(logits, targets)
        if self.cldice_weight > 0:
            probs = torch.sigmoid(logits)
            cl_loss = soft_cldice(probs, targets, iters=self.cldice_iters)
            return ft_loss + self.cldice_weight * cl_loss
        return ft_loss


# ============================================================================
# VALIDATION METRIC -- FIXED to global (micro-averaged) Dice
# ============================================================================
# WHAT WAS WRONG: the previous version computed one Dice score PER validation
# patch, then averaged those per-patch scores (a "macro" average):
#     scores.extend(((2*inter+eps)/(denom+eps)).tolist())
#     return mean(scores)
# With val_pos_patch_prob previously at 0.5, roughly half the fixed validation
# patches were near-empty-of-filament random crops. On an EMPTY-ground-truth
# patch, a model that correctly predicts nothing scores dice ~= eps/eps ~= 1.0
# -- a "free" near-perfect score unrelated to real segmentation skill.
# Averaging those trivial scores in with genuine ones inflated the reported
# metric well above real performance (measured on a matched synthetic
# example: 0.72 macro-average vs. 0.45 on positive-content patches only).
#
# THE FIX: sum intersection and sum(pred)+sum(gt) across the ENTIRE
# validation set FIRST, then divide ONCE at the end (global/micro average).
# An empty-GT patch then contributes 0 to both the numerator and denominator
# -- it literally cannot inflate the score just for being trivially correct.
# This is also more directly comparable to how the real leaderboard metric
# behaves: it doesn't hand out free credit per-image for predicting nothing
# on an image that happens to have nothing in it.
@torch.no_grad()
def compute_val_dice(model, loader, device, threshold=0.5, eps=1e-7):
    model.eval()
    inter_total, pred_total, gt_total = 0.0, 0.0, 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        preds = (torch.sigmoid(model(images)) >= threshold).float()
        inter_total += (preds * masks).sum().item()
        pred_total += preds.sum().item()
        gt_total += masks.sum().item()
    return (2 * inter_total + eps) / (pred_total + gt_total + eps)

## 4. Training Loop 🔁¶
Trains on full-resolution random patches biased toward filament-containing crops. Checkpoints are
selected on the corrected (global/micro-averaged) validation Dice.

In [6]:
# ============================================================================
# 4. TRAINING LOOP -- transforms
# ============================================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Training: NO resize. Pad-if-needed (for source images smaller than the patch)
# plus augmentations, since sample_patch() already extracted a native-resolution
# patch_size x patch_size crop.
train_transform = A.Compose([
    A.PadIfNeeded(min_height=CFG['patch_size'], min_width=CFG['patch_size'], border_mode=cv2.BORDER_REFLECT),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Fixed-patch validation transform: NO resize (avoids the pixel-deletion problem
# resizing to a small fixed size causes for thin filaments) and deliberately NO
# random augmentation -- validation should measure the model on a fixed,
# unperturbed view so Val Dice changes reflect the model improving, not a
# different random flip/rotation landing differently.
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

print("✅ Transforms loaded successfully!")

✅ Transforms loaded successfully!


In [7]:
# ============================================================================
# 4. TRAINING LOOP -- dataloaders, model, optimizer, loop
# ============================================================================
from torch.utils.data import DataLoader
from tqdm import tqdm

train_dataset = SolarDataset(train_df, TRAIN_IMG_DIR, annotations_data, train_transform,
                             CFG['patch_size'], CFG['pos_patch_prob'])

# NEW: val_pos_patch_prob (0.85, matching training) instead of the previous hardcoded
# 0.5 -- see compute_val_dice's docstring in the model/loss cell for why that
# combination (low pos_patch_prob + per-image score averaging) inflated Val Dice.
val_dataset = FixedPatchSolarDataset(val_df, TRAIN_IMG_DIR, annotations_data, val_transform,
                                     CFG['patch_size'], CFG['seed'], pos_patch_prob=CFG['val_pos_patch_prob'])

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(CFG['seed']), persistent_workers=CFG['num_workers']>0)
val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker, persistent_workers=CFG['num_workers']>0)

model = build_model().to(CFG['device'])
criterion = CombinedLoss(cldice_weight=CFG['cldice_weight'])
warmup_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CFG['bce_pos_weight']], device=CFG['device']))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)

# Halve LR if Val Dice doesn't improve for CFG['lr_patience'] epochs -- targets the
# oscillation seen when LR stayed constant the whole way through a run.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=CFG['lr_factor'], patience=CFG['lr_patience'])
scaler = torch.cuda.amp.GradScaler(enabled=CFG['use_amp'])

best_dice=-float('inf'); best_path='best_model.pt'; accum_steps=CFG['accum_steps']

print(f"⚙️  batch_size={CFG['batch_size']} x accum_steps={accum_steps} = effective batch "
      f"{CFG['batch_size']*accum_steps} | AMP={CFG['use_amp']} | grad_clip={CFG['clip_grad_norm']} "
      f"| freeze_bn={CFG['freeze_bn']} | val_pos_patch_prob={CFG['val_pos_patch_prob']}")

for epoch in range(CFG['epochs']):
    is_warmup = epoch < CFG['warmup_epochs']
    active_criterion = warmup_criterion if is_warmup else criterion

    # ---- Training phase ----
    model.train()
    # model.train() just reset EVERY submodule (including BatchNorm) back to training
    # mode -- re-apply the freeze here, every epoch, or it silently stops working
    # after epoch 1.
    if CFG['freeze_bn']:
        model.apply(set_bn_eval)

    optimizer.zero_grad(set_to_none=True); train_loss=0.0
    for step,(images,masks) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1} Train')):
        images=images.to(CFG['device'], non_blocking=True); masks=masks.to(CFG['device'], non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):
            logits=model(images); raw_loss=active_criterion(logits,masks); loss=raw_loss/accum_steps
        scaler.scale(loss).backward(); train_loss += raw_loss.detach().item()
        if (step+1)%accum_steps==0 or (step+1)==len(train_loader):
            if CFG['clip_grad_norm'] and CFG['clip_grad_norm']>0:
                # Gradients must be unscaled BEFORE clipping under AMP, or the clip
                # threshold is compared against artificially-scaled-up gradient values.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad_norm'])
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)

    # ---- Validation phase ----
    model.eval()
    val_dice=compute_val_dice(model,val_loader,CFG['device'],threshold=CFG['threshold'])
    # Note: scheduler.step() is called every epoch (including warm-up) here, unlike an
    # earlier version of this loop that skipped it during warm-up. Val Dice is a
    # comparable, well-defined metric in both phases (unlike train loss, which uses a
    # different loss function during warm-up), so there's no reason to exclude those
    # epochs from the plateau-patience count.
    scheduler.step(val_dice)
    lr_now=optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1}/{CFG["epochs"]} | Train Loss: {train_loss/max(len(train_loader),1):.4f} | Val Dice: {val_dice:.4f} | LR: {lr_now:.2e}')

    if val_dice < 1e-4:
        print("   ⚠️  Val Dice ~0 -- model predicting (near) all-background. Check the mask "
              "sanity-check cell output, then try raising CFG['bce_pos_weight'] or CFG['warmup_epochs'].")
    elif val_dice > 0.999:
        print("   ⚠️  Val Dice ~1.0 -- suspiciously perfect, check for a train/val leak.")

    if val_dice>best_dice:
        best_dice=val_dice
        torch.save({'model_state_dict':model.state_dict(),'epoch':epoch+1,'val_dice':best_dice,'cfg':CFG},best_path)
        print(f'  Saved best checkpoint: {best_path} (Dice={best_dice:.4f})')

if os.path.exists(best_path):
    ckpt=torch.load(best_path,map_location=CFG['device'])
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}, Dice={ckpt["val_dice"]:.4f}')
    print('NOTE: this Dice number now uses the corrected global/micro-averaged metric on')
    print('      pos_patch_prob=0.85 validation patches -- it will likely read LOWER than the')
    print("      previous run's numbers (which were inflated ~1.6x by the averaging bug), even")
    print('      if the model itself is just as good or better. Compare trends, not raw values,')
    print('      against the previous run.')

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

⚙️  batch_size=2 x accum_steps=4 = effective batch 8 | AMP=True | grad_clip=1.0 | freeze_bn=True | val_pos_patch_prob=0.85


Epoch 1 Train: 100%|██████████| 283/283 [02:11<00:00,  2.15it/s]


Epoch 1/25 | Train Loss: 1.3070 | Val Dice: 0.0343 | LR: 3.00e-04
  Saved best checkpoint: best_model.pt (Dice=0.0343)


...

Epoch 23/25 | Train Loss: nan | Val Dice: 0.6366 | LR: 1.87e-05


Epoch 24 Train: 100%|██████████| 283/283 [02:14<00:00,  2.11it/s]


Epoch 24/25 | Train Loss: nan | Val Dice: 0.6366 | LR: 1.87e-05


Epoch 25 Train: 100%|██████████| 283/283 [02:13<00:00,  2.11it/s]


Epoch 25/25 | Train Loss: nan | Val Dice: 0.6366 | LR: 9.37e-06
Loaded best checkpoint from epoch 5, Dice=0.6622
NOTE: this Dice number now uses the corrected global/micro-averaged metric on
      pos_patch_prob=0.85 validation patches -- it will likely read LOWER than the
      previous run's numbers (which were inflated ~1.6x by the averaging bug), even
      if the model itself is just as good or better. Compare trends, not raw values,
      against the previous run.


## 5. Instance-Level RLE Encoding & Submission Generation 📄¶
Splits each predicted mask into individual filament instances via connected components, runs
inference over overlapping full-resolution tiles + TTA, and guarantees every test image appears
at least once.

In [8]:
# ============================================================================
# 5. INSTANCE-LEVEL RLE ENCODING & SUBMISSION GENERATION
# ============================================================================
import scipy.ndimage as ndi

def instances_from_prob_mask(probs, threshold=0.40, min_pixel_size=30, close_kernel=3):
    """Threshold -> optional morphological closing (reconnects barbs broken by a thin
    gap) -> connected components -> one instance mask per component, dropping tiny
    noise blobs below min_pixel_size."""
    binary = (probs >= threshold).astype(np.uint8)
    if close_kernel and close_kernel > 1:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    labels, n = ndi.label(binary)
    masks = []
    for label_id in range(1, n + 1):
        m = (labels == label_id).astype(np.uint8)
        if int(m.sum()) >= min_pixel_size:
            masks.append(m)
    return masks


class SolarTestDataset(Dataset):
    """Full native-resolution test image, untouched -- tiling happens inside
    sliding_window_predict, one tile at a time, matching what the model was trained on."""
    def __init__(self, df, img_dir, transform=None):
        self.df, self.img_dir, self.transform = df.reset_index(drop=True), img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        filename = self.df.iloc[idx]["filename"]
        image = cv2.imread(str(Path(self.img_dir) / filename))
        if image is None:
            raise FileNotFoundError(f"Could not read {filename}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, filename


test_dataset = SolarTestDataset(df_test, TEST_IMG_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

submissions = []
images_with_zero_detections = []
model.eval()

print("🔮 Generating Predictions for Test Set (full-resolution sliding-window + TTA)...")

with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc="Generating Submission"):
        images = images.to(CFG["device"])
        probs = sliding_window_predict(model, images[0], tile_size=CFG["patch_size"], overlap=128, device=CFG["device"]).cpu().numpy()[0, 0]
        instance_masks = instances_from_prob_mask(
            probs, threshold=CFG["threshold"], min_pixel_size=30, close_kernel=3
        )
        img_id = Path(filenames[0]).stem
        if not instance_masks:
            # Guarantees every test image_id appears at least once -- a version of this
            # notebook without this scored 0.00 despite otherwise-correct code, strongly
            # suggesting the grader requires full image coverage and hard-fails on any
            # missing image_id rather than ignoring it.
            submissions.append({"filament_id": f"{img_id}_1", "segmentation_rle": mask_to_coco_rle(np.zeros(probs.shape, dtype=np.uint8))})
            images_with_zero_detections.append(img_id)
        else:
            for idx, inst_mask in enumerate(instance_masks, 1):
                submissions.append({"filament_id": f"{img_id}_{idx}", "segmentation_rle": mask_to_coco_rle(inst_mask)})

# Explicit columns even if submissions ends up empty -- pd.DataFrame([]) with no rows
# has ZERO columns, not just zero rows, which caused a real "column not found" grader
# error on an earlier run.
df_sub = pd.DataFrame(submissions, columns=["filament_id", "segmentation_rle"])
df_sub.to_csv("submission.csv", index=False)

n_unique_images = df_sub["filament_id"].apply(lambda s: s.rsplit("_", 1)[0]).nunique()
print("submission.csv created successfully!")
print(f"Total rows: {len(df_sub)} | Images with no detections: {len(images_with_zero_detections)}")
print(f"Unique image_ids covered: {n_unique_images} / {len(df_test)}")
assert n_unique_images == len(df_test), (
    "Every test image should appear at least once -- if this fires, some image was "
    "skipped entirely and will likely score like a previous 0.00 submission did."
)

🔮 Generating Predictions for Test Set (full-resolution sliding-window + TTA)...


Generating Submission: 100%|██████████| 180/180 [36:01<00:00, 12.01s/it]

submission.csv created successfully!
Total rows: 4416 | Images with no detections: 0
Unique image_ids covered: 180 / 180
